# Qwen Server for DeepDive Intelligence Integration

This notebook runs a FastAPI server exposing Qwen 2.5 7B Instruct (quantized in 4-bit) as a public endpoint via ngrok. It connects to your local DeepDive Intelligence backend to provide LLM assessment augmentation.

## Step 1: Install Packages

Install the required libraries for running Qwen (with GPU support) and setting up the API.

In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q fastapi uvicorn pyngrok nest_asyncio pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.6 MB/s eta 0:00:00:00:0100:01


## Step 2: Load Qwen Model

Initialize the tokenizer and load Qwen 2.5 7B Instruct using 4-bit quantization to fit on Colab's T4 GPU.

In [2]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

print("Qwen Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen Loaded


In [3]:
messages = [
    {
        "role": "user",
        "content": "Why might Iran and the United States resume negotiations?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Why might Iran and the United States resume negotiations?
assistant
Iran and the United States may consider resuming negotiations for several reasons:

1. **Sanctions Relief**: The lifting of certain sanctions could be a significant incentive for both sides to return to the negotiating table. Economic relief can help alleviate financial pressures on Iran and improve the economic situation in the U.S., especially in sectors that have been heavily impacted by these measures.

2. **Regional Stability**: There is a mutual interest in regional stability, particularly in light of ongoing conflicts and tensions in the Middle East. Resolving issues related to Iran's nuclear program could contribute to a more stable region, which benefits both countries and their allies.

3. **Diplomatic Relations**: Improving diplomatic relations can enhance cooperation on various international issues. For example, both nations ha

In [4]:
messages = [
    {
        "role": "user",
        "content": "Why might Iran and the United States resume negotiations?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Why might Iran and the United States resume negotiations?
assistant
Iran and the United States may resume negotiations for several reasons, including:

1. **Strategic Interests**: Both countries have strategic interests that could be mutually beneficial through negotiation. For example, the U.S. may seek to stabilize the region or address global security concerns, while Iran may want to ensure its nuclear program is recognized as peaceful and gain relief from economic sanctions.

2. **Economic Factors**: Sanctions imposed on Iran by the U.S. and other countries have had significant economic impacts. Resuming negotiations could lead to the lifting of some sanctions, which would benefit Iran's economy. Similarly, the U.S. could also benefit from increased trade with Iran.

3. **Regional Stability**: Negotiations can contribute to regional stability. By reducing tensions and addressing issues such as terroris

In [5]:
from pyngrok import ngrok

ngrok.set_auth_token(
    "2yorKv3tTYbmq9BxW2x43SsdzCz_V7WDR8i3paJQuanVCU64"
)

print("Token Added")

Token Added                                                                                         


In [6]:
from fastapi import FastAPI

app = FastAPI()

In [8]:
@app.post("/generate")
async def generate(data: dict):

    prompt = f"""
Goal:
{data["goal"]}

Assessment:
{data["assessment"]}

Improve this intelligence assessment.
"""

    messages = [
        {
            "role": "system",
            "content": "You are an intelligence analyst."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return {
        "answer": response
    }

In [9]:
!pip install pyngrok nest_asyncio fastapi uvicorn

In [10]:
public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://61e0-34-16-194-167.ngrok-free.app" -> "http://localhost:8000"


In [12]:
app

In [14]:
@app.get("/")
async def home():
    return {"status": "running"}

@app.post("/generate")
async def generate(data: dict):
    return {
        "message": "API working",
        "received": data
    }

In [15]:
import nest_asyncio
nest_asyncio.apply()


In [ ]:
import uvicorn
from threading import Thread

def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

server = Thread(target=run)
server.start()

INFO:     Started server process [337]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2405:201:6040:b8df:7998:fd3c:f9dd:be55:0 - "GET / HTTP/1.1" 200 OK
INFO:     2405:201:6040:b8df:7998:fd3c:f9dd:be55:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2405:201:6040:b8df:7998:fd3c:f9dd:be55:0 - "GET / HTTP/1.1" 200 OK
INFO:     2405:201:6040:b8df:7998:fd3c:f9dd:be55:0 - "GET / HTTP/1.1" 200 OK
INFO:     2405:201:6040:b8df:7998:fd3c:f9dd:be55:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2405:201:6040:b8df:7998:fd3c:f9dd:be55:0 - "GET /openapi.json HTTP/1.1" 200 OK


/usr/local/lib/python3.12/dist-packages/fastapi/openapi/utils.py:252: UserWarning: Duplicate Operation ID generate_generate_post for function generate
  warnings.warn(message, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/fastapi/openapi/utils.py:252: UserWarning: Duplicate Operation ID home__get for function home
  warnings.warn(message, stacklevel=1)


INFO:     2405:201:6040:b8df:7998:fd3c:f9dd:be55:0 - "POST /generate HTTP/1.1" 200 OK


In [11]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

uvicorn.run(
    app,
    host="0.0.0.0",
    port=8000
)

RuntimeError: asyncio.run() cannot be called from a running event loop